# CITADEL

## Scidocs

In [ ]:
cd dpr-scale

bash dataset_prepare.sh
bash generate_embeddings.sh
bash merge_embeddings.sh
bash prune_embeddings.sh
bash quantize_embeddings.sh
bash Retrieval.sh
bash Evaluation.sh

In [ ]:
cd dpr-scale

bash beir_pipe_scidocs.sh

In [ ]:
$BASE_DIR/
├── embeddings/scidocs/               
├── merged_embeddings/scidocs/
│   ├── expert/                        
│   ├── expert_pruned0.5/
│   ├── expert_pruned0.5_pq_nbits2/
│   ├── expert_pruned0.7/
│   └── ...
├── retrieval/scidocs/
│   ├── pruned_0.5/
│   │   ├── retrieval.trec
│   │   └── performance/performance.yaml
│   ├── pruned_0.7/
│   └── ...
└── results/scidocs/
    ├── pruned_0.5/
    │   └── eval_results.txt
    ├── pruned_0.7/
    └── ...

In [ ]:
import os
import yaml
import csv
import re
from glob import glob

# 配置路径
BASE_DIR = "/data1/chenyifeng/MultiVector-Backup/dpr-scale"
DATASET = "scidocs"
pruning_weights = [0.5, 0.7, 0.9, 1.1, 1.3]

def parse_eval_results(file_path):
    """从 eval_results.txt 提取所有指标，返回字典"""

    metrics = {}
    if not os.path.exists(file_path):
        return metrics
    with open(file_path, 'r') as f:
        content = f.read()

    # 匹配形如 "NDCG@10: 0.1234" 的行
    pattern = re.compile(r'(\w+)@(\d+):\s+([\d.]+)')
    for match in pattern.finditer(content):
        metric_name = match.group(1).lower()
        k = match.group(2)
        value = float(match.group(3))
        key = f"{metric_name}_{k}"   # 例如 ndcg_10, map_100, recall_10, p_100
        metrics[key] = value
    return metrics

def parse_performance(file_path):
    """从 performance.yaml 读取所有字段，返回字典"""

    if not os.path.exists(file_path):
        return {}
    with open(file_path, 'r') as f:
        data = yaml.safe_load(f)
    return data

def main():
    all_rows = []
    for w in pruning_weights:
        # 构建路径（权重值中的点转换为下划线，如0.5 -> 0_5）
        weight_str = str(w).replace('.', '_')
        eval_file = os.path.join(BASE_DIR, "results", DATASET, f"pruned_{weight_str}", "eval_results.txt")
        perf_file = os.path.join(BASE_DIR, "retrieval", DATASET, f"pruned_{weight_str}", "performance", "performance.yaml")

        # 解析数据
        metrics = parse_eval_results(eval_file)
        perf = parse_performance(perf_file)

        # 合并数据（添加标识字段）
        row = {
            "dataset": DATASET,
            "split": "test",
            "baseline": "citadel",
            "pruning_weight": w,
            **perf,          # 展开 performance 中的所有键值
            **metrics        # 展开评估指标
        }
        all_rows.append(row)

    if not all_rows:
        print("No data found. Please check paths and pruning_weights.")
        return

    # 收集所有出现的列名（自动获取所有键的并集）
    all_fieldnames = set()
    for row in all_rows:
        all_fieldnames.update(row.keys())

    # 按顺序排列，将标识字段放在前面
    sorted_fields = ["dataset", "split", "baseline", "pruning_weight"]
    
    # 添加 performance 和 metrics 的其他字段，按字母排序
    other_fields = sorted([f for f in all_fieldnames if f not in sorted_fields])
    fieldnames = sorted_fields + other_fields

    # 写入 CSV
    output_dir = os.path.join(BASE_DIR, "results", DATASET)
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "summary.csv")

    with open(output_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"Summary CSV saved to: {output_file}")
    print(f"Columns: {fieldnames}")

if __name__ == "__main__":
    main()